# Word Embeddings and Representation Learning
## Progression from Frequency-Based to Contextualized Embeddings

**Course:** Natural Language Processing  
**Task:** Comparing three fundamental approaches to word representation in NLP

### Assignment Overview
This comprehensive assignment explores the evolution of word embeddings:
1. **Module 1:** Frequency-Based Embeddings (BoW, TF-IDF)
2. **Module 2:** Prediction-Based Embeddings (Word2Vec, GloVe)
3. **Module 3:** Contextualized Embeddings (BERT, ELMo)

**Dataset:** Twitter Sentiment Classification  
**Objective:** Understand differences between embedding types, observe performance variations, and explain results clearly

## Section 1: Data Loading and Preprocessing

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import time

# Set random seeds for reproducibility
np.random.seed(42)
import random
random.seed(42)

print("Libraries imported successfully!")

### Load Twitter Sentiment Dataset
We'll use the Twitter Sentiment dataset for classification. This dataset contains tweets labeled with sentiment (positive/negative)

In [ ]:
# Load the Twitter sentiment dataset
# Using a small sample for demonstration - can be replaced with full dataset
url = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/twitter_samples.zip"

# Alternative: Create a sample dataset for demonstration
data = {
    'text': [
        "I love this amazing product! Best purchase ever!",
        "This is terrible and a waste of money",
        "Great customer service, very helpful team",
        "Horrible experience, never again",
        "The quality is excellent and delivery was fast",
        "Worst experience of my life",
        "I'm so happy with this purchase!",
        "Complete disaster, total disappointment",
        "Fantastic! Exceeded all my expectations",
        "Absolutely disgusting, really bad",
        "Love the new design, very modern",
        "Poor quality, stopped working after one day",
        "Incredible value for money",
        "Overpriced and not worth it",
        "Amazing customer support, helped me immediately",
        "Rude staff, very unprofessional",
        "Best service in the industry",
        "Never have I been so disappointed",
        "Highly recommend to everyone!",
        "Total waste of my time and money",
        "Perfect! Exactly what I needed",
        "Defective product, asked for refund",
        "Love everything about this brand",
        "The worst purchase I've ever made",
        "Exceptional quality and fast shipping"
    ] * 4,  # Repeat to get more samples
    'sentiment': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1] * 4
}

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(f"\nSentiment distribution:\n{df['sentiment'].value_counts()}")
print(f"\nFirst 5 samples:\n{df.head()}")

### Text Preprocessing
Implement tokenization, lowercasing, stopword removal, and vocabulary creation

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

def preprocess_text(text):
    """
    Preprocess text by:
    1. Lowercasing
    2. Removing special characters and numbers
    3. Tokenization
    4. Stopword removal
    """
    # Lowercase
    text = text.lower()
    
    # Remove URLs, mentions, hashtags
    text = re.sub(r'http\S+|www\S+|@\w+|#\w+', '', text)
    
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
    
    return tokens

# Apply preprocessing
df['tokens'] = df['text'].apply(preprocess_text)

print("Preprocessing complete!")
print(f"\nExample preprocessing:")
print(f"Original: {df['text'].iloc[0]}")
print(f"Tokens: {df['tokens'].iloc[0]}")

### Train-Test Split
Split data into training (70%), validation (15%), and test (15%) sets

In [ ]:
# Split data: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    df['text'], df['sentiment'], test_size=0.30, random_state=42, stratify=df['sentiment']
)

# Split temp into 50% validation and 50% test (15% each of original)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Training set: {len(X_train)} samples ({len(X_train)/len(df)*100:.1f}%)")
print(f"Validation set: {len(X_val)} samples ({len(X_val)/len(df)*100:.1f}%)")
print(f"Test set: {len(X_test)} samples ({len(X_test)/len(df)*100:.1f}%)")

# Preprocess all sets
X_train_tokens = X_train.apply(preprocess_text)
X_val_tokens = X_val.apply(preprocess_text)
X_test_tokens = X_test.apply(preprocess_text)

print("\nPreprocessing complete for all sets!")

---

## Section 2: Module 1 - Frequency-Based Embeddings (BoW & TF-IDF)

### Overview
Frequency-based embeddings represent text as numerical vectors based on word occurrences:
- **Bag of Words (BoW):** Counts of word occurrences
- **TF-IDF:** Weighted counts considering term importance and document rarity

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# Initialize vectorizers for BoW and TF-IDF
bow_vectorizer = CountVectorizer(max_features=500, stop_words='english')
tfidf_vectorizer = TfidfVectorizer(max_features=500, stop_words='english')

# Fit vectorizers on training data and transform all sets
print("Creating Bag of Words representations...")
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_val_bow = bow_vectorizer.transform(X_val)
X_test_bow = bow_vectorizer.transform(X_test)

print(f"BoW vector shape: {X_train_bow.shape}")
print(f"BoW vocabulary size: {len(bow_vectorizer.vocabulary_)}")

print("\nCreating TF-IDF representations...")
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"TF-IDF vector shape: {X_train_tfidf.shape}")
print(f"TF-IDF vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

### Section 3: Model Training and Evaluation - Frequency-Based Methods

In [ ]:
def train_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, model_name, model):
    """
    Train and evaluate a classifier
    """
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Predictions
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)
    
    # Metrics
    results = {
        'Model': model_name,
        'Train Accuracy': accuracy_score(y_train, y_pred_train),
        'Val Accuracy': accuracy_score(y_val, y_pred_val),
        'Test Accuracy': accuracy_score(y_test, y_pred_test),
        'Precision': precision_score(y_test, y_pred_test),
        'Recall': recall_score(y_test, y_pred_test),
        'F1-Score': f1_score(y_test, y_pred_test),
        'Train Time (s)': train_time
    }
    
    return results, model, y_pred_test

# Train classifiers with BoW
print("=" * 70)
print("TRAINING CLASSIFIERS WITH BAG OF WORDS")
print("=" * 70)

lr_bow = LogisticRegression(max_iter=1000, random_state=42)
results_bow_lr, model_bow_lr, pred_bow_lr = train_and_evaluate(
    X_train_bow, X_val_bow, X_test_bow, y_train, y_val, y_test,
    "Logistic Regression + BoW", lr_bow
)

nb_bow = MultinomialNB()
results_bow_nb, model_bow_nb, pred_bow_nb = train_and_evaluate(
    X_train_bow, X_val_bow, X_test_bow, y_train, y_val, y_test,
    "Naive Bayes + BoW", nb_bow
)

print("\n" + "=" * 70)
print("TRAINING CLASSIFIERS WITH TF-IDF")
print("=" * 70)

# Train classifiers with TF-IDF
lr_tfidf = LogisticRegression(max_iter=1000, random_state=42)
results_tfidf_lr, model_tfidf_lr, pred_tfidf_lr = train_and_evaluate(
    X_train_tfidf, X_val_tfidf, X_test_tfidf, y_train, y_val, y_test,
    "Logistic Regression + TF-IDF", lr_tfidf
)

nb_tfidf = MultinomialNB()
results_tfidf_nb, model_tfidf_nb, pred_tfidf_nb = train_and_evaluate(
    X_train_tfidf, X_val_tfidf, X_test_tfidf, y_train, y_val, y_test,
    "Naive Bayes + TF-IDF", nb_tfidf
)

# Compile results
freq_results = pd.DataFrame([results_bow_lr, results_bow_nb, results_tfidf_lr, results_tfidf_nb])
print("\n" + "=" * 70)
print("FREQUENCY-BASED METHODS COMPARISON")
print("=" * 70)
print(freq_results.to_string(index=False))

### Analysis: Limitations of Frequency-Based Methods
Frequency-based embeddings have significant limitations:

1. **No semantic understanding:** Similar words (e.g., "good" and "excellent") have completely different vectors
2. **Curse of dimensionality:** With large vocabularies, vectors become very sparse and high-dimensional
3. **Loss of word order:** "dog bites man" and "man bites dog" produce identical BoW representations
4. **Poor context handling:** Cannot distinguish polysemy (words with multiple meanings)
5. **Memory intensive:** Sparse representations require significant memory for large vocabularies

---

## Section 4: Module 2 - Prediction-Based Embeddings (Word2Vec & GloVe)

### Overview
Prediction-based embeddings learn dense, continuous vector representations by predicting context words or being predicted from context. These capture semantic relationships between words.

In [ ]:
from gensim.models import Word2Vec
import numpy as np

# Prepare tokenized data for Word2Vec training
print("Training Word2Vec embeddings...")
print("-" * 70)

# Convert token lists to format required by Word2Vec
train_sentences = [tokens for tokens in X_train_tokens]

# Train Word2Vec model (Skip-gram with embedding dimension = 100)
w2v_model = Word2Vec(
    sentences=train_sentences,
    vector_size=100,        # Embedding dimension
    window=5,               # Context window size
    min_count=2,            # Minimum word frequency
    workers=4,              # Parallel processing
    sg=1,                   # Skip-gram (1) vs CBOW (0)
    epochs=10
)

print(f"Word2Vec model trained!")
print(f"Vocabulary size: {len(w2v_model.wv)}")
print(f"Embedding dimension: {w2v_model.vector_size}")

# Example: Word similarity
print("\nWord Similarity Examples (Word2Vec):")
test_words = ['love', 'hate', 'good', 'bad']
for word in test_words:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=3)
        print(f"\nWords similar to '{word}':")
        for similar_word, similarity in similar:
            print(f"  {similar_word}: {similarity:.4f}")

In [ ]:
def get_embeddings_from_tokens(token_list, model, embedding_dim):
    """
    Convert token lists to average word embeddings
    """
    embeddings = []
    for tokens in token_list:
        word_vectors = []
        for token in tokens:
            if token in model.wv:
                word_vectors.append(model.wv[token])
        
        # Use average of word vectors as document embedding
        if word_vectors:
            avg_embedding = np.mean(word_vectors, axis=0)
        else:
            avg_embedding = np.zeros(embedding_dim)
        
        embeddings.append(avg_embedding)
    
    return np.array(embeddings)

# Generate embeddings for all sets using Word2Vec
print("Generating document embeddings from Word2Vec...")
X_train_w2v = get_embeddings_from_tokens(X_train_tokens, w2v_model, 100)
X_val_w2v = get_embeddings_from_tokens(X_val_tokens, w2v_model, 100)
X_test_w2v = get_embeddings_from_tokens(X_test_tokens, w2v_model, 100)

print(f"Word2Vec embeddings shape: {X_train_w2v.shape}")

# Train classifiers with Word2Vec embeddings
print("\n" + "=" * 70)
print("TRAINING CLASSIFIERS WITH WORD2VEC EMBEDDINGS")
print("=" * 70)

lr_w2v = LogisticRegression(max_iter=1000, random_state=42)
results_w2v_lr, model_w2v_lr, pred_w2v_lr = train_and_evaluate(
    X_train_w2v, X_val_w2v, X_test_w2v, y_train, y_val, y_test,
    "Logistic Regression + Word2Vec", lr_w2v
)

from sklearn.neural_network import MLPClassifier
nn_w2v = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=1000, random_state=42)
results_w2v_nn, model_w2v_nn, pred_w2v_nn = train_and_evaluate(
    X_train_w2v, X_val_w2v, X_test_w2v, y_train, y_val, y_test,
    "Neural Network + Word2Vec", nn_w2v
)

# Compile all results
prediction_based_results = pd.DataFrame([results_w2v_lr, results_w2v_nn])
print("\n" + "=" * 70)
print("PREDICTION-BASED METHODS RESULTS")
print("=" * 70)
print(prediction_based_results.to_string(index=False))

### Section 5: Embedding Visualization with PCA and t-SNE

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Visualize Word2Vec embeddings using PCA
print("Applying PCA for dimensionality reduction...")
pca = PCA(n_components=2, random_state=42)
word_vectors_pca = pca.fit_transform(list(w2v_model.wv.vectors))

print("Applying t-SNE for dimensionality reduction (this may take a moment)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(w2v_model.wv)-1))
word_vectors_tsne = tsne.fit_transform(list(w2v_model.wv.vectors))

# Plot visualizations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA visualization
axes[0].scatter(word_vectors_pca[:, 0], word_vectors_pca[:, 1], alpha=0.6, s=30)
axes[0].set_title('Word Embeddings - PCA Visualization', fontsize=14, fontweight='bold')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].grid(True, alpha=0.3)

# Annotate some important words
important_words = ['good', 'bad', 'love', 'hate', 'amazing', 'terrible', 'excellent', 'awful']
for word in important_words:
    if word in w2v_model.wv:
        idx = list(w2v_model.wv.index_to_key).index(word)
        axes[0].annotate(word, (word_vectors_pca[idx, 0], word_vectors_pca[idx, 1]),
                        fontsize=9, alpha=0.8)

# t-SNE visualization
axes[1].scatter(word_vectors_tsne[:, 0], word_vectors_tsne[:, 1], alpha=0.6, s=30)
axes[1].set_title('Word Embeddings - t-SNE Visualization', fontsize=14, fontweight='bold')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].grid(True, alpha=0.3)

# Annotate some important words
for word in important_words:
    if word in w2v_model.wv:
        idx = list(w2v_model.wv.index_to_key).index(word)
        axes[1].annotate(word, (word_vectors_tsne[idx, 0], word_vectors_tsne[idx, 1]),
                        fontsize=9, alpha=0.8)

plt.tight_layout()
plt.show()

print("\nVisualization complete! Notice how semantically similar words cluster together.")

---

## Section 6: Module 3 - Contextualized Embeddings (BERT)

### Overview
Contextualized embeddings generate different representations for the same word depending on its context. BERT (Bidirectional Encoder Representations from Transformers) is a powerful pre-trained model that captures context from both directions.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

print("Loading pre-trained BERT model...")
print("This may take a moment on first run...")

# Load BERT model and tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name, output_hidden_states=True)
bert_model.eval()

print(f"BERT model loaded: {model_name}")
print(f"Model hidden size: {bert_model.config.hidden_size}")

def get_bert_embeddings(texts, tokenizer, model):
    """
    Extract BERT embeddings for texts (using [CLS] token representation)
    """
    embeddings = []
    
    for text in texts:
        # Tokenize and encode
        inputs = tokenizer(text, return_tensors='pt', truncation=True, 
                          max_length=128, padding=True)
        
        # Get embeddings
        with torch.no_grad():
            outputs = model(**inputs)
            # Use [CLS] token embedding (first token) for sentence representation
            cls_embedding = outputs.last_hidden_state[:, 0, :].numpy()[0]
        
        embeddings.append(cls_embedding)
    
    return np.array(embeddings)

print("\nExtracting BERT embeddings from training data (first 50 samples for speed)...")
# Use subset for speed
subset_size = min(50, len(X_train))
X_train_bert = get_bert_embeddings(X_train.iloc[:subset_size].tolist(), tokenizer, bert_model)
X_val_bert = get_bert_embeddings(X_val.tolist(), tokenizer, bert_model)
X_test_bert = get_bert_embeddings(X_test.tolist(), tokenizer, bert_model)

print(f"BERT embeddings shape: {X_train_bert.shape}")
print("BERT embeddings extraction complete!")

### Section 7: Contextualized Model Training and Evaluation

In [ ]:
print("=" * 70)
print("TRAINING CLASSIFIERS WITH BERT EMBEDDINGS")
print("=" * 70)

# Note: We only use subset of training data due to computational constraints
# In practice, use full dataset
y_train_subset = y_train.iloc[:subset_size]

lr_bert = LogisticRegression(max_iter=1000, random_state=42)
results_bert_lr, model_bert_lr, pred_bert_lr = train_and_evaluate(
    X_train_bert, X_val_bert, X_test_bert, y_train_subset, y_val, y_test,
    "Logistic Regression + BERT", lr_bert
)

nn_bert = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=1000, random_state=42)
results_bert_nn, model_bert_nn, pred_bert_nn = train_and_evaluate(
    X_train_bert, X_val_bert, X_test_bert, y_train_subset, y_val, y_test,
    "Neural Network + BERT", nn_bert
)

contextualized_results = pd.DataFrame([results_bert_lr, results_bert_nn])
print("\n" + "=" * 70)
print("CONTEXTUALIZED METHODS RESULTS")
print("=" * 70)
print(contextualized_results.to_string(index=False))

### Section 8: Polysemy Analysis - Context Effects on Word Embeddings

In [ ]:
def get_word_embedding_from_context(text, target_word, tokenizer, model):
    """
    Extract embedding for a specific word from BERT in context
    """
    # Tokenize the text
    inputs = tokenizer(text, return_tensors='pt', truncation=True, 
                      max_length=128, padding=True)
    
    # Get token IDs
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    # Get embeddings
    with torch.no_grad():
        outputs = model(**inputs)
        hidden_states = outputs.last_hidden_state[0]  # [seq_len, hidden_dim]
    
    # Find target word token(s)
    embeddings = []
    for idx, token in enumerate(tokens):
        # Match tokens that contain the target word
        if target_word.lower() in token.lower():
            embeddings.append(hidden_states[idx].numpy())
    
    return embeddings

# Analyze polysemy with the word "bank" (financial institution vs river bank)
print("\n" + "=" * 70)
print("POLYSEMY ANALYSIS: Word 'bank' in Different Contexts")
print("=" * 70)

polysemous_word = "bank"

contexts = [
    "I went to the bank to withdraw some money from my account.",
    "We sat on the bank of the river and enjoyed the sunset.",
    "The bank provided loans for small businesses in the area.",
    "Children played by the river bank during summer holidays."
]

print(f"\nAnalyzing different meanings of '{polysemous_word}':\n")

embeddings_per_context = []
for i, context in enumerate(contexts, 1):
    embeddings = get_word_embedding_from_context(context, polysemous_word, tokenizer, bert_model)
    if embeddings:
        avg_embedding = np.mean(embeddings, axis=0)
        embeddings_per_context.append(avg_embedding)
        print(f"Context {i}: {context}")
        print(f"  -> Extracted {len(embeddings)} token(s) for '{polysemous_word}'")
        print(f"  -> Embedding shape: {avg_embedding.shape}\n")

# Compute similarity between different contexts
if len(embeddings_per_context) >= 2:
    print("-" * 70)
    print("Cosine Similarity between contexts:")
    from sklearn.metrics.pairwise import cosine_similarity
    
    similarity_matrix = cosine_similarity(embeddings_per_context)
    
    for i in range(len(similarity_matrix)):
        for j in range(i+1, len(similarity_matrix)):
            sim = similarity_matrix[i][j]
            print(f"Context {i+1} vs Context {j+1}: {sim:.4f}")
    
    print("\nObservation:")
    print("Contexts 1-3 (financial): Should have higher similarity")
    print("Contexts 2,4 (river): Should have higher similarity")
    print("Financial vs River contexts: Should have lower similarity")
    print("\nThis demonstrates how contextualized embeddings capture")
    print("different semantic meanings of the same word in different contexts!")

---

## Section 9: Comprehensive Comparison of All Methods

In [ ]:
print("\n" + "=" * 70)
print("CROSS-MODULE COMPARISON: ALL EMBEDDING APPROACHES")
print("=" * 70)

# Combine all results
all_results = pd.concat([
    freq_results,
    prediction_based_results,
    contextualized_results
], ignore_index=True)

print("\nComplete Results Summary:")
print(all_results.to_string(index=False))

# Calculate average scores by method
print("\n" + "-" * 70)
print("PERFORMANCE BY EMBEDDING METHOD")
print("-" * 70)

methods = {
    'Frequency-Based (BoW)': freq_results[freq_results['Model'].str.contains('BoW')],
    'Frequency-Based (TF-IDF)': freq_results[freq_results['Model'].str.contains('TF-IDF')],
    'Prediction-Based (Word2Vec)': prediction_based_results,
    'Contextualized (BERT)': contextualized_results
}

comparison_results = []
for method_name, method_results in methods.items():
    avg_accuracy = method_results['Test Accuracy'].mean()
    avg_f1 = method_results['F1-Score'].mean()
    avg_time = method_results['Train Time (s)'].mean()
    
    comparison_results.append({
        'Method': method_name,
        'Avg Test Accuracy': f"{avg_accuracy:.4f}",
        'Avg F1-Score': f"{avg_f1:.4f}",
        'Avg Train Time (s)': f"{avg_time:.4f}"
    })

comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.to_string(index=False))

### Visualization: Performance Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Test Accuracy by Model
models = all_results['Model'].tolist()
accuracies = all_results['Test Accuracy'].tolist()
colors = ['#FF6B6B' if 'BoW' in m else '#4ECDC4' if 'TF-IDF' in m else 
          '#45B7D1' if 'Word2Vec' in m else '#FFA07A' for m in models]

axes[0, 0].barh(models, accuracies, color=colors)
axes[0, 0].set_xlabel('Test Accuracy', fontweight='bold')
axes[0, 0].set_title('Test Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0, 0].grid(axis='x', alpha=0.3)

# Plot 2: F1-Score by Model
f1_scores = all_results['F1-Score'].tolist()
axes[0, 1].barh(models, f1_scores, color=colors)
axes[0, 1].set_xlabel('F1-Score', fontweight='bold')
axes[0, 1].set_title('F1-Score Comparison', fontsize=12, fontweight='bold')
axes[0, 1].grid(axis='x', alpha=0.3)

# Plot 3: Precision vs Recall
precision = all_results['Precision'].tolist()
recall = all_results['Recall'].tolist()
axes[1, 0].scatter(recall, precision, s=200, c=range(len(models)), cmap='viridis', alpha=0.6)
for i, model in enumerate(models):
    axes[1, 0].annotate(f"{i}", (recall[i], precision[i]), fontsize=8)
axes[1, 0].set_xlabel('Recall', fontweight='bold')
axes[1, 0].set_ylabel('Precision', fontweight='bold')
axes[1, 0].set_title('Precision vs Recall', fontsize=12, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Plot 4: Training Time
train_times = all_results['Train Time (s)'].tolist()
axes[1, 1].barh(models, train_times, color=colors)
axes[1, 1].set_xlabel('Training Time (seconds)', fontweight='bold')
axes[1, 1].set_title('Training Time Comparison', fontsize=12, fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualization complete!")

---

## Section 10: Comprehensive Analysis and Conclusions

### Key Findings and Trade-offs

#### 1. **Frequency-Based Embeddings (BoW & TF-IDF)**
**Advantages:**
- Fast and computationally efficient
- Easy to understand and interpret
- No training required (can be used directly)
- Low memory footprint

**Disadvantages:**
- Cannot capture semantic meaning or word relationships
- Ignores word order and context
- Suffers from sparsity (high-dimensional, mostly zeros)
- Cannot handle polysemy (multiple meanings)
- TF-IDF vocabulary size grows with corpus

#### 2. **Prediction-Based Embeddings (Word2Vec)**
**Advantages:**
- Dense, low-dimensional representations
- Captures semantic and syntactic relationships
- Efficient computation on large datasets
- Meaningful in vector space (e.g., king - man + woman ≈ queen)

**Disadvantages:**
- Requires large amount of data to train well
- Static representations (no context sensitivity)
- Cannot handle out-of-vocabulary words in inference
- Still cannot distinguish polysemy well

#### 3. **Contextualized Embeddings (BERT)**
**Advantages:**
- Context-aware representations
- Captures both semantic and syntactic information
- Handles polysemy effectively (same word, different contexts, different embeddings)
- Pre-trained on massive corpora (transfer learning)
- State-of-the-art performance on most NLP tasks

**Disadvantages:**
- Computationally expensive (requires GPUs for efficiency)
- Longer inference time
- Complex to understand and debug
- Large model size
- Requires fine-tuning for specific tasks

### Evolution of Word Representations

```
┌─────────────────────────────────────────────────────────────────┐
│  WORD REPRESENTATION PROGRESSION IN NLP                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. FREQUENCY-BASED (1950s-2000s)                              │
│     └─ Count words, ignore meaning                              │
│     └─ Sparse, high-dimensional, static                         │
│                                                                 │
│  2. PREDICTION-BASED (2010s)                                   │
│     └─ Learn from context prediction task                       │
│     └─ Dense, low-dimensional, semantic similarity              │
│     └─ Still static (same word = same embedding)                │
│                                                                 │
│  3. CONTEXTUALIZED (2015-present)                              │
│     └─ Model context bidirectionally                            │
│     └─ Dynamic (context-dependent representations)              │
│     └─ Captures subtle linguistic nuances                       │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Performance Characteristics

| Characteristic | BoW/TF-IDF | Word2Vec | BERT |
|---|---|---|---|
| **Semantic Awareness** | ❌ | ✅ | ✅✅✅ |
| **Context Sensitivity** | ❌ | ❌ | ✅✅✅ |
| **Polysemy Handling** | ❌ | ❌ | ✅✅✅ |
| **Training Speed** | ✅✅✅ | ✅✅ | ❌ |
| **Inference Speed** | ✅✅✅ | ✅✅ | ❌ |
| **Memory Efficiency** | ❌ | ✅✅ | ❌ |
| **Interpretability** | ✅✅✅ | ✅ | ❌ |
| **Performance on Sentiment** | ✅✅ | ✅✅✅ | ✅✅✅✅ |

### When to Use Each Method

**Use Frequency-Based Methods When:**
- Computational resources are limited
- You need fast training and inference
- Interpretability is critical
- Dataset is small
- Baseline comparison is needed

**Use Prediction-Based Methods When:**
- You have substantial training data
- Semantic similarity matters
- You want balance between performance and efficiency
- Context is relatively consistent

**Use Contextualized Methods When:**
- State-of-the-art performance is required
- Context heavily influences meaning
- You have access to computational resources
- Fine-tuning for specific task is acceptable
- Transfer learning can be leveraged

### Conclusion

This assignment demonstrated the complete evolution of word embeddings in NLP:

1. **Frequency-based methods** provide a foundation and demonstrate why simple counting is insufficient for understanding language semantics.

2. **Prediction-based methods** represent a major breakthrough, learning meaningful relationships between words through neural prediction tasks.

3. **Contextualized methods** achieve state-of-the-art results by understanding that the meaning of words depends critically on their context.

**The Journey Forward:**
The field continues to evolve with newer architectures like Transformers (GPT, T5), large language models (LLMs), and multimodal models. Understanding this progression is crucial for appreciating modern NLP techniques and making informed choices about which representation method to use.

**Key Takeaways:**
- There is no one-size-fits-all solution; choose based on your task, resources, and requirements
- Performance typically improves with more sophisticated methods, but at computational cost
- The ability to handle context and capture semantic meaning is critical for language understanding
- Transfer learning with pre-trained models has become the dominant paradigm